# Lab 2.2 - Train the Sample Problem with TensorFlow/Keras (IMU-based gesture recognition)

This notebook has `# TODO` markers (TODO 1-5, with TODO 4 split into 4a-4e).
Work through them in order - later cells depend on earlier ones (e.g. the
hyperparameter sweep in Step 2 reuses the `build_model()` function you
complete in `src/cnn2d_model.py` for TODO 2).

The Magic Wand dataset shared in this repo (`dataset/negative`,
`dataset/ring`, `dataset/slope`, `dataset/wing`) is small. To train a model
that generalizes well, you need more data. Before starting this notebook:

1. Build the `data_collection` firmware and flash it onto the Pico (see
   `data_collection/README.md`).
2. Record several sessions per gesture class (negative/ring/slope/wing),
   holding the Pico with the IMU oriented as described in the top-level
   `README.md` (x right, y back, z up).
3. Copy the recorded `.txt` files off the SD card, together with copies of
   the matching files from the shared `dataset/<class_name>/` folders, into
   `dataset/custom/<class_name>/` - so each of the four class folders under
   `dataset/custom/` ends up containing both the shared recordings and your
   own.

Once `dataset/custom/` is populated with all four classes, you're ready for
Step 0 below.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Make the helper modules in src/ importable (preprocessor.py, cnn2d_model.py)
SRC_DIR = str(Path.cwd() / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

tf.random.set_seed(10)
np.random.seed(10)

print("TensorFlow version:", tf.__version__)

## Step 0 - Prepare the dataset

Unlike MNIST, there's no `tf.keras.datasets` loader for this problem - you
collected the raw recordings yourself, so you first need to turn them into
a fixed-length numpy array the model can train on. `src/preprocessor.py`
already implements the full pipeline (load CSV, convert to g, filter,
resample to a fixed length, split, normalize); you just need to call it.

**TODO 1:** Call `prepare_dataset(target_length=128)` to preprocess every
`.txt` recording under `dataset/custom/<class_name>/` and cache the result
as a `.npz` file at `DEFAULT_OUTPUT_PATH` (pass `target_length=128` so the
saved sequences match the `(128, 3, 1)` input shape `build_model()` expects
in Step 1). Then call `load_prepared_dataset(DEFAULT_OUTPUT_PATH)` to load
it back as a dict of `(X, y)` pairs per split. You only need to call
`prepare_dataset(...)` again if you add more recordings later.

In [ ]:
from src.preprocessor import DEFAULT_OUTPUT_PATH, prepare_dataset, load_prepared_dataset

# TODO 1: call prepare_dataset(...) once to preprocess and cache the dataset,
# then load it back with load_prepared_dataset(...)
dataset = None

X_train, y_train = dataset["train"]
X_validation, y_validation = dataset["validation"]
X_test, y_test = dataset["test"]

# Conv2D expects a trailing channel dimension: (time, 3) -> (time, 3, 1)
X_train = X_train[..., np.newaxis]
X_validation = X_validation[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("X_train:", X_train.shape, " X_validation:", X_validation.shape,
      " X_test:", X_test.shape)
print("Classes:", dataset["class_names"])

## Step 1 - Build and train a small CNN

Architecture (complete this in `src/cnn2d_model.py`, not in the notebook -
you'll import `build_model()` both here and again in Step 2):

`Input (time, 3, 1) -> Conv2D(num_filters1, kernel=(4,3), padding="same", activation) -> MaxPool2D((3,3), strides=(3,3), padding="same") -> Conv2D(num_filters2, kernel=(4,1), padding="same", activation) -> MaxPool2D((3,1), strides=(3,1), padding="same") -> Flatten -> Dropout(dropout) -> Dense(dense_units, activation) -> Dropout(dropout) -> Dense(num_classes, softmax)`

Default hyperparameters: `num_filters1=8, num_filters2=16, dense_units=16,
activation="relu", dropout=0.1` - these give the baseline model; you'll vary
them in Step 2's sweep.

A couple of things worth knowing before you write this:
- The first conv/pool block convolves across both time and the 3
  acceleration axes; by the time the second conv runs, the axis dimension
  has already been pooled down to 1, so the second conv only convolves
  along time (hence the `(4, 1)` kernel instead of `(4, 3)`).
- Every op used here (`Conv2D`, `MaxPooling2D`, `Flatten`, `Dropout`,
  `Dense`) maps onto a kernel included in TFLite Micro's standard op
  resolver - this matters once the model gets deployed to the Pico.
- `input_shape` and `num_classes` are *not* hyperparameters to sweep - pass
  them in from the data (`X_train.shape[1:]` and
  `len(dataset["class_names"])`), don't hardcode numbers.

**TODO 2:** Open `src/cnn2d_model.py` and complete `build_model()` following
the architecture above.

In [ ]:
def build_model(input_shape, num_classes):
    model = None
    
    return model

model = build_model(
    input_shape=X_train.shape[1:],
    num_classes=len(dataset["class_names"]),
)
model.summary()

**TODO 3:** Compile and train the model.
- Optimizer: `"adam"`
- Loss: sparse categorical crossentropy (labels are integer class indices,
  not one-hot)
- Metric: accuracy
- Train for enough epochs to see the validation curve flatten out (try ~30
  as a starting point), batch size 16, using `(X_validation, y_validation)`
  as validation data.

In [ ]:
# TODO 3: compile and fit the model, capturing the returned History object
history = None

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Baseline model -- test accuracy: {test_acc:.4f}, test loss: {test_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline CNN training curve")
plt.legend()
plt.tight_layout()
plt.savefig("baseline_training_curve.png", dpi=110)
plt.show()

## Step 2 - Experiment with training parameters

Try changing each of the following, one at a time, and note the effect on
accuracy, training time, and model size:

- Learning rate
- Optimizer (SGD vs Adam vs RMSprop)
- Number of epochs
- Batch size
- Activation functions
- Number of filters/neurons per layer (`num_filters1`, `num_filters2`,
  `dense_units`)
- Dropout

**TODO 4:** Complete `run_experiment()` below (build, train, and evaluate a
model, timing the training run and counting its trainable parameters), then
add your own entries to the `results` list - at least one experiment per
bullet point above. Reuse `build_model()` and `run_experiment()`; don't
rewrite the training loop by hand for each one.

In [ ]:
def count_params(m):
    return int(np.sum([tf.keras.backend.count_params(w) for w in m.trainable_weights]))

def run_experiment(name, optimizer, epochs=20, batch_size=16, activation="relu",
                    num_filters1=8, num_filters2=16, dense_units=16, dropout=0.1):
    # TODO 4a: build the model with the given hyperparameters
    m = None

    # TODO 4b: compile it (same loss/metric as Step 1) with the given optimizer

    # TODO 4c: time a call to m.fit() with the given epochs/batch_size
    # (verbose=0), using (X_validation, y_validation) as validation data
    start = time.time()
    elapsed = None

    # TODO 4d: evaluate on the test set and count trainable params
    acc = None
    n_params = None

    return {
        "experiment": name,
        "test_accuracy": round(acc, 4),
        "train_time_sec": round(elapsed, 1),
        "trainable_params": n_params,
    }

# TODO 4e: add at least one run_experiment(...) call per bullet point above
results = []
results.append(run_experiment("baseline (adam, lr=default)", optimizer="adam"))

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(9, 4))
plt.barh(results_df["experiment"], results_df["test_accuracy"])
plt.xlabel("Test accuracy")
plt.title("Hyperparameter sweep -- test accuracy by configuration")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("hyperparameter_sweep.png", dpi=110)
plt.show()

best_row = results_df.loc[results_df["test_accuracy"].idxmax()]
print("Best configuration in this sweep:")
print(best_row)

## Step 3 - Save the trained model

**TODO 5:** Save the model in both `.h5` and `SavedModel` format.
- `model.save("magic_wand_cnn.h5")` saves the legacy HDF5 format.
- `model.export("magic_wand_cnn_savedmodel")` exports a `SavedModel` (Keras
  3's `model.save()` no longer writes `SavedModel` for an extension-less
  path the way `tf_keras.Model.save()` does in the MNIST notebook - use
  `.export(...)` here instead).

Keep both files/folders safe if you continue on to quantization or
on-device deployment.

In [ ]:
# TODO 5: save the model as magic_wand_cnn.h5 and as a SavedModel export


## Discussion Topics

Answer in a few sentences each:

1. Which single hyperparameter gave you the biggest accuracy gain (or loss)?
2. How much did adding your own collected recordings (vs. only the small
   shared dataset) change validation/test accuracy? What does that tell you
   about generalization?
3. What's the tradeoff between model size and accuracy in your experiments?